In [1]:
import pandas as pd
import sqlite3

orders = pd.read_csv("store_orders.csv")
customers = pd.read_csv("store_customers.csv")

con = sqlite3.connect("store.db")

orders.to_sql("orders", con, if_exists="replace", index=False)
customers.to_sql("customers", con, if_exists="replace", index=False)

40

In [2]:
pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", con)

,name
0,orders
1,customers


In [3]:
pd.read_sql("SELECT * FROM orders LIMIT 5;", con)

,order_id,customer_id,product,category,amount,order_date
0,5000,4,paper,office,14.09,2026-05-25
1,5001,15,planner,office,51.86,2026-05-08
2,5002,34,shelf,furniture,116.25,2026-06-03
3,5003,2,granola,snacks,19.72,2026-05-19
4,5004,17,monitor,electronics,284.80,2026-05-10


In [4]:
pd.read_sql("""
SELECT *
FROM orders
WHERE amount > 100
ORDER BY amount DESC;
""", con)

,order_id,customer_id,product,category,amount,order_date
0,5153,20,dock,electronics,499.00,2026-05-25
1,5140,30,headset,electronics,295.36,2026-05-26
2,5047,901,headset,electronics,293.79,2026-05-15
3,5150,903,monitor,electronics,293.35,2026-06-06
4,5014,6,monitor,electronics,287.83,2026-05-22
...,...,...,...,...,...,...
79,5107,38,shelf,furniture,110.31,2026-05-17
80,5046,13,shelf,furniture,108.47,2026-05-28
81,5009,14,chair,furniture,107.20,2026-05-24
82,5007,36,dock,electronics,103.77,2026-05-21


In [5]:
pd.read_sql("""
SELECT category,
       COUNT(*) AS order_count,
       ROUND(AVG(amount), 2) AS avg_amount
FROM orders
GROUP BY category
ORDER BY avg_amount DESC;
""", con)

,category,order_count,avg_amount
0,electronics,60,190.57
1,furniture,46,146.25
2,office,51,30.13
3,snacks,43,14.62


In [6]:
pd.read_sql("""
SELECT o.order_id,
       o.customer_id,
       c.name,
       c.city,
       o.product,
       o.category,
       o.amount,
       o.order_date
FROM orders AS o
JOIN customers AS c
ON o.customer_id = c.customer_id
LIMIT 10;
""", con)

,order_id,customer_id,name,city,product,category,amount,order_date
0,5000,4,Dev Patel,Los Angeles,paper,office,14.09,2026-05-25
1,5001,15,Omar Garcia,Santa Monica,planner,office,51.86,2026-05-08
2,5002,34,Nina Patel,Venice,shelf,furniture,116.25,2026-06-03
3,5003,2,Ben Kim,Culver City,granola,snacks,19.72,2026-05-19
4,5004,17,Quinn Silva,Santa Monica,monitor,electronics,284.80,2026-05-10
5,5005,16,Priya Okafor,Culver City,monitor,electronics,66.10,2026-06-10
6,5006,27,Grace Silva,Culver City,headset,electronics,86.67,2026-06-08
7,5007,36,Priya Okafor,Culver City,dock,electronics,103.77,2026-05-21
8,5008,30,Jon Brooks,Los Angeles,headset,electronics,267.39,2026-06-08
9,5009,14,Nina Patel,Inglewood,chair,furniture,107.20,2026-05-24


In [8]:
step5_sql = """
SELECT c.city,
       ROUND(SUM(o.amount), 2) AS total_spend
FROM orders AS o
JOIN customers AS c
ON o.customer_id = c.customer_id
GROUP BY c.city
ORDER BY total_spend DESC;
"""

step5_df = pd.read_sql(step5_sql, con)
step5_df

,city,total_spend
0,Los Angeles,5862.99
1,Culver City,5316.88
2,Santa Monica,4017.93
3,Inglewood,2718.25
4,Torrance,1037.25
5,Venice,462.36


In [9]:
print("Orders:", len(orders))

joined_rows = pd.read_sql("""
SELECT o.*
FROM orders AS o
JOIN customers AS c
ON o.customer_id = c.customer_id;
""", con)

print("Joined rows:", len(joined_rows))

Orders: 200
Joined rows: 195


There are 200 orders in the original table, but the join has 195 rows. Five orders are missing because their customer IDs are not found in the customers table.

import matplotlib.pyplot as plt

step5_df.plot(
    x="city",
    y="total_spend",
    kind="bar",
    legend=False
)

plt.title("Total Spend by City")
plt.xlabel("City")
plt.ylabel("Total Spend ($)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

Business Question: Which product was ordered the most times?

In [13]:
pd.read_sql("""
SELECT product, COUNT(*) AS order_count
FROM orders
GROUP BY product
ORDER BY order_count DESC
LIMIT 1;
""", con)

,product,order_count
0,headset,23


The headset was ordered the most. It appeared in 23 orders.

I would not include customer names when sharing the results. I would use general or grouped data instead to keep customer information private.